# Accrual Quality (OLS baseline)

**Purpose:** Construct working capital accruals and estimate the McNichols (2002)
accrual regression using OLS with a rolling window.

This notebook produces:
1. The WCA dataset with all scaled regressors
2. `sigma_OLS` — the OLS-based noise measure (Robustness 1)

The Hierarchical Bayesian estimator (primary method) is in Notebook 03.

**Key formula:**
$$\frac{WCA_{i,t}}{A_{i,t-1}} = \alpha_i + \beta_1 \frac{CFO_{t-1}}{A} + \beta_2 \frac{CFO_t}{A} + \beta_3 \frac{CFO_{t+1}^*}{A} + \beta_4 \frac{\Delta REV_t}{A} + \beta_5 \frac{PPE_t}{A} + \epsilon_{i,t}$$

$^*CFO_{t+1} = NA$ for portfolio year $t$ — model marginalises over unknown future cash flow.

In [ ]:
import sys
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

notebook_dir = Path().resolve()
sys.path.insert(0, str(notebook_dir.parents[0] / "src"))

from config import PROCESSED_DIR
from accrual_model import build_wca, compute_ols_rmse

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load merged data

In [ ]:
data = pd.read_parquet(PROCESSED_DIR / "data_merged.parquet")
print(f'Loaded {len(data):,} firm-years, {data["Ticker"].nunique()} tickers')
data.head(3)

## 2. Construct WCA and scaled regressors

In [ ]:
wca_data = build_wca(data)

print(f'WCA dataset: {len(wca_data):,} firm-years')
print(f'Scaled variables summary:')
wca_data[['WCA_scaled', 'CFO_scaled', 'dREV_scaled', 'PPE_scaled']].describe().round(4)

## 3. Visual check — WCA and CFO relationship

Per Dechow-Dichev logic, WCA and CFO should be negatively correlated (contemporaneous matching).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# WCA distribution
axes[0].hist(wca_data['WCA_scaled'].dropna().clip(-1, 1),
             bins=80, edgecolor='none', color='steelblue', alpha=0.8)
axes[0].set_title('Distribution of WCA / AT_lag')
axes[0].set_xlabel('WCA scaled')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='black', lw=0.8, linestyle='--')

# WCA vs CFO scatter
sample = wca_data.sample(min(3000, len(wca_data)), random_state=42)
axes[1].scatter(sample['CFO_scaled'].clip(-1, 1),
                sample['WCA_scaled'].clip(-1, 1),
                alpha=0.25, s=8, color='steelblue')
axes[1].set_title('WCA vs CFO (scaled) — expect negative slope')
axes[1].set_xlabel('CFO / AT_lag')
axes[1].set_ylabel('WCA / AT_lag')

# Add regression line
valid = sample[['CFO_scaled', 'WCA_scaled']].dropna()
m, b = np.polyfit(valid['CFO_scaled'], valid['WCA_scaled'], 1)
xs = np.linspace(valid['CFO_scaled'].min(), valid['CFO_scaled'].max(), 100)
axes[1].plot(xs, m * xs + b, color='firebrick', lw=1.5,
             label=f'slope = {m:.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../output/wca_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

corr = wca_data[['WCA_scaled', 'CFO_scaled']].corr().iloc[0, 1]
print(f'WCA / CFO correlation: {corr:.4f} (expect negative)')

## 4. OLS McNichols — rolling window

Rolling window: expanding 3–5 years startup, then 5-year rolling.

`sigma_OLS` = RMSE of in-sample residuals (Robustness 1 measure).

Note: RMSE, not std — captures both magnitude and variability of residuals.

In [ ]:
ols_results = compute_ols_rmse(
    wca_data,
    min_train_years=3,
    max_train_years=5
)

print(f'OLS results: {len(ols_results):,} firm-years')
ols_results.describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# sigma_OLS distribution
axes[0].hist(ols_results['sigma_OLS'].dropna().clip(0, 0.3),
             bins=60, edgecolor='none', color='steelblue', alpha=0.8)
axes[0].set_title('Distribution of sigma_OLS (RMSE)')
axes[0].set_xlabel('sigma_OLS')
axes[0].set_ylabel('Count')

# Annual median sigma_OLS
annual_sigma = ols_results.groupby('Year')['sigma_OLS'].median()
axes[1].plot(annual_sigma.index, annual_sigma.values,
             marker='o', ms=4, color='steelblue')
axes[1].set_title('Median sigma_OLS by year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median RMSE')
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('../output/sigma_ols_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save outputs

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Full WCA dataset (used as input to HB model in notebook 03)
wca_data.to_parquet('../data/processed/wca_data.parquet', index=False)

# OLS sigma (Robustness 1)
ols_results.to_parquet('../data/processed/sigma_ols.parquet', index=False)

print('Saved:')
print('  data/processed/wca_data.parquet')
print('  data/processed/sigma_ols.parquet')
print(f'  {len(wca_data):,} firm-years in WCA dataset')
print(f'  {len(ols_results):,} firm-years with sigma_OLS')